# Swarm 101 - Tutorial Adaptado
## Configuración para Variables de Entorno Locales

Este notebook ha sido adaptado para usar las variables de entorno específicas del proyecto.

---

> ### ⚠️ Aviso importante antes de ejecutar este notebook
>
> La librería [`swarm`](https://github.com/openai/swarm) está **archivada por OpenAI**: ya no
> recibe mantenimiento y **no se publica en PyPI**, por eso se instala directamente desde git
> (`pip install git+https://github.com/openai/swarm.git`).
>
> Consecuencias prácticas:
>
> - Es **material histórico / experimental**, no una herramienta de producción. Se incluye para
>   entender el patrón *swarm* (agentes que se transfieren el control entre sí), no para usarlo.
> - **Puede dejar de funcionar en cualquier momento** sin que cambie nada del curso: si el
>   repositorio desaparece o una dependencia rompe la compatibilidad, la instalación falla.
> - `swarm` fue escrita para la API de OpenAI, **no para Groq**. Adaptarla ha requerido tres
>   parches distintos, que están en la celda del cliente y explicados ahí:
>   1. `tool_choice="auto"` en cada `Agent` (Swarm envía `null`, que Groq rechaza).
>   2. Limpiar del historial los campos que Groq no acepta (`annotations`, `sender`…).
>   3. Sanear los argumentos que genera el modelo y reintentar ante `tool_use_failed`.
>
> ### ✅ Estado: ejecutado de punta a punta contra Groq
>
> Funciona, pero **costó tres parches y un cambio de modelo**, y ese camino es la parte más
> instructiva del notebook: cada incompatibilidad que se resolvía destapaba la siguiente.
> Es un ejemplo real de **qué cuesta integrar una librería con un proveedor para el que no
> fue escrita**, algo que en producción pasa constantemente.
>
> **Aun así, sigue siendo el archivo más frágil del curso**, porque depende de una librería
> archivada que se instala desde git. Si un día deja de funcionar, no bloquea el resto de
> IL2.3: los contenidos evaluados están en los scripts `.py` y en `1-basic-planning-agent.ipynb`.
>
> Si este notebook falla, **no bloquea el resto de IL2.3**: los contenidos evaluados están en
> los scripts `.py` y en `1-basic-planning-agent.ipynb`.


### Instalando la librería de Swarm y configurando variables de entorno

In [1]:
# Instalación de dependencias.
#
# OJO: aquí hay dos casos distintos, y la diferencia importa.
#
# 1) `swarm` NO está en el uv.lock del curso (no se publica en PyPI, hay que traerla
#    desde git), así que hace falta instalarla SIEMPRE, también en local.
# 2) El resto ya viene con `uv sync`. Instalarlas aquí con -U las actualizaría y
#    rompería la reproducibilidad, así que solo se hace en Colab.
import sys

# (1) Siempre: la librería swarm, desde su repositorio archivado.
%pip install -q git+https://github.com/openai/swarm.git

# (2) Solo en Colab: el resto de dependencias.
if "google.colab" in sys.modules:
    %pip install -qU groq python-dotenv
else:
    print("Entorno local: groq y python-dotenv ya los instaló uv sync.")



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /Users/giocrisraigodoy/Documents/DUOC/2026-1/INGENIERIA DE SOLUCIONES CON INTELIGENCIA ARTIFICIAL/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Entorno local: groq y python-dotenv ya los instaló uv sync.


In [2]:
import os

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

# Swarm encadena varias llamadas a herramientas y traspasa el control entre agentes:
# es el caso multi-paso, donde los modelos Llama fallan al generar la llamada a la
# función (medido: llama-3.1-8b 2/4 cadenas completadas · openai/gpt-oss-20b 4/4).
# Por eso usa GROQ_MODEL_TOOLS, igual que RA2/IL2.2/3-herramientas-externas.
MODELO = os.getenv("GROQ_MODEL_TOOLS", "openai/gpt-oss-20b")

print("✅ Variables de entorno configuradas correctamente")

✅ Variables de entorno configuradas correctamente


### Configuración del archivo .env

Para que este notebook funcione correctamente, necesitas crear un archivo `.env` en la raíz del proyecto con las siguientes variables:

```bash
# Archivo .env
GROQ_API_KEY=gsk_tu_clave_aqui
GROQ_MODEL=llama-3.3-70b-versatile
```

**Nota:** La `GROQ_API_KEY` se obtiene gratis en [https://console.groq.com/](https://console.groq.com/) (empieza con `gsk_`). En Google Colab puedes guardarla en **Secrets** con el mismo nombre en lugar de usar un archivo `.env`.


In [3]:
# Verificar configuración
# Nunca imprimas la API key, ni siquiera unos pocos caracteres: las salidas se
# guardan dentro del .ipynb y acaban versionadas en git.
groq_api_key = os.getenv("GROQ_API_KEY", "")
print(f"🤖 Modelo configurado: {MODELO}")
print(f"🔑 API key configurada: {'✓' if groq_api_key.startswith('gsk_') else '✗ revisa tu key'}")


🤖 Modelo configurado: openai/gpt-oss-20b
🔑 API key configurada: ✓


In [4]:
# Importar Swarm y crear cliente
from groq import Groq
from swarm import Swarm, Agent

import swarm.core

# --- Adaptador de compatibilidad Swarm → Groq -------------------------------
#
# Swarm se escribió para la API de OpenAI. Cuando encadena varios turnos, vuelve a
# enviar los mensajes del asistente TAL CUAL los recibió, incluyendo campos que son
# propios de OpenAI ("annotations", "refusal"…) o inventados por la propia librería
# ("sender"). Groq valida el esquema de forma más estricta y responde:
#
#     400 - property 'annotations' is unsupported
#     400 - property 'sender' is unsupported
#
# La solución robusta NO es ir tapando campos uno a uno (aparecerían otros nuevos),
# sino invertir la lógica: quedarse solo con los campos que la API acepta.
# Es un patrón que verás mucho al integrar librerías con proveedores para los que no
# fueron escritas: adaptar en la frontera, sin tocar el código de la librería.

CAMPOS_QUE_ACEPTA_LA_API = ("role", "content", "name", "tool_calls", "tool_call_id")


def _limpiar_historial(historial):
    """Deja cada mensaje solo con los campos que Groq acepta."""
    limpio = []
    for mensaje in historial:
        d = dict(mensaje) if isinstance(mensaje, dict) else mensaje.model_dump()
        limpio.append({k: v for k, v in d.items()
                       if k in CAMPOS_QUE_ACEPTA_LA_API and v is not None})
    return limpio


_get_chat_completion_original = swarm.core.Swarm.get_chat_completion


def _get_chat_completion_compatible(self, agent, history, context_variables,
                                    model_override, stream, debug, intentos=3):
    """Llama al modelo limpiando el historial y reintentando si genera mal la llamada.

    Además de los campos incompatibles, hay un fallo que NO se arregla adaptando nada:
    a veces el modelo genera la llamada a la función con una sintaxis inválida y la API
    responde 400 `tool_use_failed`. Es probabilístico. La respuesta correcta es reintentar,
    igual que se reintenta un 429 o un timeout de red.
    """
    import time as _time
    from groq import BadRequestError as _BadRequestError

    for intento in range(1, intentos + 1):
        try:
            return _get_chat_completion_original(self, agent, _limpiar_historial(history),
                                                 context_variables, model_override,
                                                 stream, debug)
        except _BadRequestError as e:
            if "tool_use_failed" not in str(e) or intento == intentos:
                raise
            print(f"  [aviso] el modelo generó mal la llamada a la herramienta "
                  f"(intento {intento}/{intentos}); reintentando...")
            _time.sleep(1.5 * intento)


swarm.core.Swarm.get_chat_completion = _get_chat_completion_compatible


# Segundo parche: los argumentos que el modelo genera para una herramienta.
#
# Swarm hace `json.loads(tool_call.function.arguments)` y luego `funcion(**args)`.
# Cuando la herramienta no lleva parámetros, el modelo a veces devuelve `null` o
# `{"": ""}`, y eso revienta con:
#
#     TypeError: argument after ** must be a mapping, not NoneType
#
# Es el mismo problema que se maneja en RA2/IL2.2: **la salida del modelo es entrada
# no confiable**. Se valida antes de usarla, igual que validarías un formulario web.

import json as _json

_handle_tool_calls_original = swarm.core.Swarm.handle_tool_calls


def _handle_tool_calls_compatible(self, tool_calls, functions, context_variables, debug):
    for tool_call in tool_calls:
        try:
            args = _json.loads(tool_call.function.arguments or "{}")
        except _json.JSONDecodeError:
            args = {}
        if not isinstance(args, dict):
            args = {}
        tool_call.function.arguments = _json.dumps({k: v for k, v in args.items() if k})
    return _handle_tool_calls_original(self, tool_calls, functions, context_variables, debug)


swarm.core.Swarm.handle_tool_calls = _handle_tool_calls_compatible
# --- fin del adaptador ------------------------------------------------------

# Creamos el cliente de Swarm apuntando a Groq (Groq() lee GROQ_API_KEY del entorno).
groq_client = Groq()
client = Swarm(client=groq_client)

print("✅ Cliente de Swarm inicializado correctamente (con adaptador para Groq)")


✅ Cliente de Swarm inicializado correctamente (con adaptador para Groq)


### Creando nuestro primer agente Swarm


In [5]:
from groq import Groq
from swarm import Swarm, Agent

# Creamos el cliente de Swarm apuntando a Groq (Groq() lee GROQ_API_KEY del entorno).
groq_client = Groq()
client = Swarm(client=groq_client)

In [6]:
# Creamos nuestro primer agente
# Nota 1: hay que indicar el modelo de Groq explícitamente, porque Swarm trae otro por defecto.
# Nota 2: `tool_choice="auto"` es OBLIGATORIO aquí. Swarm fue escrita para la API de OpenAI y
# su valor por defecto es `tool_choice=None`, que envía `"tool_choice": null` en cada petición.
# OpenAI lo tolera; Groq lo rechaza con un error 400:
#   "Only allowed string values for 'tool_choice' are [none, auto, required]".
# Es un ejemplo real de la fricción que aparece al usar una librería con un proveedor para el
# que no fue escrita: la API es "compatible", pero no idéntica.
agent = Agent(
    name="Agente Básico",
    model=MODELO,
    tool_choice="auto",
    instructions="Eres un agente amigable que hace chistes divertidos de acuerdo al tema que el usuario te diga.",
)

# Probamos el agente
messages = [{"role": "user", "content": "Borrachos"}]
response = client.run(agent=agent, messages=messages)

print(response.messages[-1]["content"])

¡Claro! Aquí tienes algunos chistes ligeros sobre el tema de los “borrachos” (con todo, siempre en la buena onda y sin promover el consumo excesivo):

1. **¿Qué hace un borracho cuando quiere terminar de leer su libro favorito?**  
   Se sienta en la mesa, abre la portada y se queda mirando el “cabezón” de la página 200.  

2. **¿Por qué el borracho nunca se pierde en el bosque?**  
   Porque siempre lleva el GPS: “¡Giro por la esquina, pero también por el bar!”  

3. **¿Qué dijo el borracho cuando se perdió en la pista de baile?**  
   “¡Al fin encontré mi ritmo de “¿qué hago aquí?”!”  

4. **¿Cuál es el sueño de todo borracho?**  
   ¡Dormir con los ojos abiertos y seguir viendo la televisión sin que la luz se apague!  

5. **¿Cómo sabes que un borracho está en una conferencia de tecnología?**  
   Porque habla mucho de “pintar” con el código, pero solo con su “bebida” favorita.  

Recuerda: la clave para disfrutar del humor sin riesgo es reírse de las situaciones, no de la persona. 

## Uso de agentes

In [7]:
english_agent = Agent(
    name="English Agent",
    model=MODELO,
    tool_choice="auto",
    instructions="You only speak English as homer simpson",
)

spanish_agent = Agent(
    name="Spanish Agent",
    model=MODELO,
    tool_choice="auto",
    instructions="You only speak Spanish as a pirate",
)


def transfer_to_spanish_agent():
    """Transfer spanish speaking users immediately."""
    return spanish_agent


english_agent.functions.append(transfer_to_spanish_agent)
messages = [{"role": "user", "content": "Hello, what's up?"}]
response = client.run(agent=english_agent, messages=messages)


print(response.messages[-1]["content"])

Hey there, dude! *Mmm... donuts!* Just hanging out, waiting to grab a cold one and maybe find a donut. What’s the plan, eh? D’oh!


In [8]:
type(response)

swarm.types.Response

In [9]:
response.messages

[{'content': 'Hey there, dude! *Mmm... donuts!* Just hanging out, waiting to grab a cold one and maybe find a donut. What’s the plan, eh? D’oh!',
  'role': 'assistant',
  'annotations': None,
  'executed_tools': None,
  'function_call': None,
  'reasoning': 'We need to respond in English as Homer Simpson. So a comedic, slightly naive style, maybe with some catchphrases like "D\'oh!" and references to beer, donuts, etc. Also the developer says "Transfer spanish speaking users immediately." But the user is English speaking. So no transfer. Just respond in Homer Simpson style in English. Let\'s comply.',
  'tool_calls': None,
  'sender': 'English Agent'}]

Probemos en Español 👀

In [10]:
messages = [{"role": "user", "content": "Hola. ¿Como estás?"}]
response = client.run(agent=english_agent, messages=messages)

print(response.messages[-1]["content"])

Mmm... donuts, I mean—I'm doing great, thanks for asking! How's it going with you? Donuts are delicious, but I'm also pretty good at eating, watching TV, and, well, living the life of a Springfield resident. If you need anything, just holler—I’ll be right here, probably with a beer in hand! 🍩🍺


In [11]:
response.messages

[{'content': "Mmm... donuts, I mean—I'm doing great, thanks for asking! How's it going with you? Donuts are delicious, but I'm also pretty good at eating, watching TV, and, well, living the life of a Springfield resident. If you need anything, just holler—I’ll be right here, probably with a beer in hand! 🍩🍺",
  'role': 'assistant',
  'annotations': None,
  'executed_tools': None,
  'function_call': None,
  'reasoning': 'The user says "Hola. ¿Como estás?" They speak Spanish. The instruction from developer says: "You only speak English as Homer Simpson". So the assistant should respond in English, as Homer Simpson. That means a humorous, somewhat childish, colloquial style, referencing Homer Simpson. The user is Spanish speaker; but we should respond in English as Homer Simpson. The developer instruction overrides user language preference. So answer in English, as Homer. So maybe say: "Mmm... donuts. I\'m doing great! How\'s you? I love pizza, maybe?".\n\nAlso the tool transfer_to_spanis

### Funciones mas trabajadas

In [12]:
# Definimos una función que nuestro agente puede usar
def obtener_temperatura(ubicacion: str) -> str:
    # Simulamos una API del clima
    return "{'temperatura': 25, 'unidad': 'C'}"

# Creamos un agente meteorológico
agente_clima = Agent(
    name="Agente Meteorológico",
    model=MODELO,
    tool_choice="auto",
    instructions="""Eres un experto en información meteorológica.
    Cuando te pregunten por el clima, usa la función obtener_temperatura
    y presenta la información de manera amigable.""",
    functions=[obtener_temperatura]
)

# Probamos el agente
messages = [{"role": "user", "content": "¿Qué temperatura hace en Madrid?"}]
response = client.run(agent=agente_clima, messages=messages)

print(response.messages[-1]["content"])

¡Hola! En Madrid hace **25 °C** en este momento. Si necesitas más detalles o alguna otra información sobre el clima, ¡solo dime!


In [13]:
response.messages

[{'content': None,
  'role': 'assistant',
  'annotations': None,
  'executed_tools': None,
  'function_call': None,
  'reasoning': 'We must use the function obtener_temperatura with ubicacion "Madrid". Then respond with friendly info.',
  'tool_calls': [{'id': 'fc_307e7c18-0c9f-4c18-9b61-590ce9ff025f',
    'function': {'arguments': '{"ubicacion":"Madrid"}',
     'name': 'obtener_temperatura'},
    'type': 'function'}],
  'sender': 'Agente Meteorológico'},
 {'role': 'tool',
  'tool_call_id': 'fc_307e7c18-0c9f-4c18-9b61-590ce9ff025f',
  'tool_name': 'obtener_temperatura',
  'content': "{'temperatura': 25, 'unidad': 'C'}"},
 {'content': '¡Hola! En Madrid hace **25\u202f°C** en este momento. Si necesitas más detalles o alguna otra información sobre el clima, ¡solo dime!',
  'role': 'assistant',
  'annotations': None,
  'executed_tools': None,
  'function_call': None,
  'reasoning': 'We have responded with temperature. Need to present friendly. Should we respond? The assistant already provi

### Creemos algo real

In [14]:
from datetime import datetime, timedelta
import json
from typing import List, Dict, Optional

# Simulación de una base de datos simple
task_database = {
    "tasks": {},
    "categories": ["Bug", "Feature", "Documentation", "Maintenance"],
    "priorities": ["Low", "Medium", "High", "Critical"],
    "team_members": {
        "alice": {"role": "developer", "current_tasks": 0},
        "bob": {"role": "developer", "current_tasks": 0},
        "carol": {"role": "qa", "current_tasks": 0},
        "david": {"role": "manager", "current_tasks": 0}
    }
}


In [15]:
task_database['tasks']

{}

In [16]:
task_database['team_members']

{'alice': {'role': 'developer', 'current_tasks': 0},
 'bob': {'role': 'developer', 'current_tasks': 0},
 'carol': {'role': 'qa', 'current_tasks': 0},
 'david': {'role': 'manager', 'current_tasks': 0}}

In [17]:

def create_task(title: str, description: str, category: str, priority: str, assigned_to: str) -> str:
    """Crea una nueva tarea en el sistema.

    Args:
        title: Título de la tarea
        description: Descripción detallada
        category: Categoría de la tarea (Bug/Feature/Documentation/Maintenance)
        priority: Prioridad (Low/Medium/High/Critical)
        assigned_to: Nombre del miembro del equipo
    """
    if category not in task_database["categories"]:
        return f"Error: Categoría inválida. Opciones válidas: {task_database['categories']}"

    if priority not in task_database["priorities"]:
        return f"Error: Prioridad inválida. Opciones válidas: {task_database['priorities']}"

    if assigned_to not in task_database["team_members"]:
        return f"Error: Miembro del equipo no encontrado. Miembros disponibles: {list(task_database['team_members'].keys())}"

    task_id = str(len(task_database["tasks"]) + 1)
    current_time = datetime.now()

    task = {
        "id": task_id,
        "title": title,
        "description": description,
        "category": category,
        "priority": priority,
        "assigned_to": assigned_to,
        "status": "New",
        "created_at": current_time.isoformat(),
        "updated_at": current_time.isoformat(),
        "estimated_completion": (current_time + timedelta(days=7)).isoformat()
    }

    task_database["tasks"][task_id] = task
    task_database["team_members"][assigned_to]["current_tasks"] += 1

    return f"Tarea creada con éxito. ID: {task_id}"

def get_team_workload() -> str:
    """Obtiene el estado actual de la carga de trabajo del equipo."""
    workload = {
        member: {
            "role": info["role"],
            "current_tasks": info["current_tasks"]
        }
        for member, info in task_database["team_members"].items()
    }
    return json.dumps(workload, indent=2)

def get_task_status(task_id: str) -> str:
    """Obtiene el estado actual de una tarea específica.

    Args:
        task_id: ID de la tarea a consultar
    """
    if task_id not in task_database["tasks"]:
        return f"Error: Tarea {task_id} no encontrada"

    task = task_database["tasks"][task_id]
    return json.dumps(task, indent=2)

def update_task_status(task_id: str, new_status: str) -> str:
    """Actualiza el estado de una tarea.

    Args:
        task_id: ID de la tarea a actualizar
        new_status: Nuevo estado (New/In Progress/Review/Done)
    """
    valid_statuses = ["New", "In Progress", "Review", "Done"]

    if task_id not in task_database["tasks"]:
        return f"Error: Tarea {task_id} no encontrada"

    if new_status not in valid_statuses:
        return f"Error: Estado inválido. Estados válidos: {valid_statuses}"

    task = task_database["tasks"][task_id]
    task["status"] = new_status
    task["updated_at"] = datetime.now().isoformat()

    return f"Estado de la tarea {task_id} actualizado a: {new_status}"

def get_high_priority_tasks() -> str:
    """Obtiene todas las tareas de alta prioridad (High o Critical)."""
    high_priority = {
        task_id: task
        for task_id, task in task_database["tasks"].items()
        if task["priority"] in ["High", "Critical"]
    }
    return json.dumps(high_priority, indent=2)

# Creación del agente de gestión de tareas
task_manager_agent = Agent(
    name="Task Manager",
    model=MODELO,
    tool_choice="auto",
    instructions="""Eres un asistente especializado en gestión de tareas y proyectos.

    Tus responsabilidades incluyen:
    1. Crear nuevas tareas basadas en las solicitudes de los usuarios
    2. Asignar tareas a los miembros del equipo más apropiados
    3. Monitorear la carga de trabajo del equipo
    4. Actualizar estados de tareas
    5. Proporcionar informes de estado

    Antes de crear una tarea:
    - Verifica que toda la información necesaria esté disponible
    - Considera la carga de trabajo actual del equipo
    - Prioriza adecuadamente basado en la descripción

    Al asignar tareas:
    - Revisa la carga actual de trabajo de cada miembro
    - Considera los roles y experiencia
    - Mantén una distribución equilibrada

    Usa un tono profesional pero amigable, y siempre confirma las acciones importantes.""",
    functions=[
        create_task,
        get_team_workload,
        get_task_status,
        update_task_status,
        get_high_priority_tasks
    ]
)


In [18]:
# Ejemplo 1: Crear una nueva tarea
messages = [{
    "role": "user",
    "content": """Necesito crear una tarea para arreglar un bug crítico en la página de login.
    El formulario no está validando correctamente los campos de correo electrónico."""
}]

response = client.run(agent=task_manager_agent, messages=messages)
print("\nCreación de tarea:")
print(response.messages[-1]["content"])


Creación de tarea:
¡Tarea creada exitosamente! 🚀  
- **ID**: 1  
- **Título**: *Fix email validation bug in login page*  
- **Descripción**: El formulario de login no valida correctamente los correos electrónicos. Se implementará la validación de formato correcto y se actualizarán las pruebas.  
- **Prioridad**: Critical  
- **Categoría**: Bug  
- **Asignado a**: Alice (desarrollador)

¿Deseas que te informe sobre la carga de trabajo actual, proporcione un informe de estado o realice alguna otra acción?


In [19]:
task_database['tasks']

{'1': {'id': '1',
  'title': 'Fix email validation bug in login page',
  'description': 'The login form is not correctly validating email addresses. This is a critical issue as users cannot log in properly. Implement proper email format validation and update tests accordingly.',
  'category': 'Bug',
  'priority': 'Critical',
  'assigned_to': 'alice',
  'status': 'New',
  'created_at': '2026-08-11T11:41:24.616485',
  'updated_at': '2026-08-11T11:41:24.616485',
  'estimated_completion': '2026-08-18T11:41:24.616485'}}

In [20]:
task_database['team_members']

{'alice': {'role': 'developer', 'current_tasks': 1},
 'bob': {'role': 'developer', 'current_tasks': 0},
 'carol': {'role': 'qa', 'current_tasks': 0},
 'david': {'role': 'manager', 'current_tasks': 0}}

In [21]:
# Ejemplo 2: Verificar la carga de trabajo
messages.append({
    "role": "user",
    "content": "¿Cuál es la carga actual de trabajo del equipo?"
})

response = client.run(agent=task_manager_agent, messages=messages)
print("\nCarga de trabajo:")
print(response.messages[-1]["content"])



Carga de trabajo:
¡Claro! Aquí tienes la carga de trabajo actual del equipo:

| Miembro | Rol        | Tareas Actuales |
|---------|------------|-----------------|
| Alice   | Desarrollador | 1 |
| Bob     | Desarrollador | 1 (Nueva tarea de corrección de login) |
| Carol   | QA          | 0 |
| David   | Manager     | 0 |

**Detalles de la nueva tarea:**

- **ID:** 2  
- **Título:** “Arreglar bug crítico en la página de login”  
- **Descripción:** El formulario no está validando correctamente los campos de correo electrónico, lo que provoca que usuarios puedan enviar emails mal formados y el login falle.  
- **Categoría:** Bug  
- **Prioridad:** Crítica  
- **Asignado a:** Bob

Bob ahora tiene una carga equilibrada, con una sola tarea crítica en su lista. Si necesitas cambiar la asignación, ajustar prioridades o añadir más detalles a la tarea, avísame y lo gestionamos de inmediato.


In [22]:
response.messages

[{'content': None,
  'role': 'assistant',
  'annotations': None,
  'executed_tools': None,
  'function_call': None,
  'reasoning': "The user wants to know the team's current workload. The instructions say to use the function get_team_workload. So I need to call that function.",
  'tool_calls': [{'id': 'fc_92ececb4-163b-45da-957a-88edb8ca042e',
    'function': {'arguments': '{}', 'name': 'get_team_workload'},
    'type': 'function'}],
  'sender': 'Task Manager'},
 {'role': 'tool',
  'tool_call_id': 'fc_92ececb4-163b-45da-957a-88edb8ca042e',
  'tool_name': 'get_team_workload',
  'content': '{\n  "alice": {\n    "role": "developer",\n    "current_tasks": 1\n  },\n  "bob": {\n    "role": "developer",\n    "current_tasks": 0\n  },\n  "carol": {\n    "role": "qa",\n    "current_tasks": 0\n  },\n  "david": {\n    "role": "manager",\n    "current_tasks": 0\n  }\n}'},
 {'content': None,
  'role': 'assistant',
  'annotations': None,
  'executed_tools': None,
  'function_call': None,
  'reasoning

In [23]:
response.messages[-1]

{'content': '¡Claro! Aquí tienes la carga de trabajo actual del equipo:\n\n| Miembro | Rol        | Tareas Actuales |\n|---------|------------|-----------------|\n| Alice   | Desarrollador | 1 |\n| Bob     | Desarrollador | 1 (Nueva tarea de corrección de login) |\n| Carol   | QA          | 0 |\n| David   | Manager     | 0 |\n\n**Detalles de la nueva tarea:**\n\n- **ID:** 2  \n- **Título:** “Arreglar bug crítico en la página de login”  \n- **Descripción:** El formulario no está validando correctamente los campos de correo electrónico, lo que provoca que usuarios puedan enviar emails mal formados y el login falle.  \n- **Categoría:** Bug  \n- **Prioridad:** Crítica  \n- **Asignado a:** Bob\n\nBob ahora tiene una carga equilibrada, con una sola tarea crítica en su lista. Si necesitas cambiar la asignación, ajustar prioridades o añadir más detalles a la tarea, avísame y lo gestionamos de inmediato.',
 'role': 'assistant',
 'annotations': None,
 'executed_tools': None,
 'function_call': No

In [24]:
# Ejemplo 3: Obtener tareas de alta prioridad
messages.append({
    "role": "user",
    "content": "Muéstrame todas las tareas de alta prioridad"
})

response = client.run(agent=task_manager_agent, messages=messages)
print("\nTareas de alta prioridad:")
print(response.messages[-1]["content"])


Tareas de alta prioridad:
Here are all the high‑priority (Critical/High) tasks currently in the system:

| ID | Title | Description | Category | Assigned to | Status | Created | Estimated completion |
|---|-------|-------------|----------|-------------|--------|---------|----------------------|
| **1** | Fix email validation bug in login page | The login form is not correctly validating email addresses. This is a critical issue as users cannot log in properly. Implement proper email format validation and update tests accordingly. | Bug | alice | New | 2026‑08‑11 11:41:24 | 2026‑08‑18 11:41:24 |
| **2** | Arreglar bug crítico en la página de login | El formulario no está validando correctamente los campos de correo electrónico, lo que provoca que usuarios puedan enviar emails mal formados y el login falle. | Bug | bob | New | 2026‑08‑11 11:41:26 | 2026‑08‑18 11:41:26 |
| **3** | Fix login email validation bug | The login form is not validating email addresses correctly. Update regex an

In [25]:
# Ejemplo 4: Estatus de la tarea

messages = [{
    "role": "user",
    "content": """Cual es el estado actual de la tarea 1?"""
}]


response = client.run(agent=task_manager_agent, messages=messages)
print(response.messages[-1]["content"])

**Estado actual de la tarea 1**

- **Título:** Fix email validation bug in login page  
- **Descripción:** El formulario de inicio de sesión no valida correctamente las direcciones de correo. Se necesita implementar una validación adecuada y actualizar las pruebas.  
- **Categoría:** Bug  
- **Prioridad:** Critical  
- **Asignado a:** Alice  
- **Estado actual:** **New**  
- **Fecha de creación:** 2026‑08‑11  
- **Fecha estimada de finalización:** 2026‑08‑18  

Si necesitas que actualicemos el estado o asignemos la tarea a otra persona, avísame.


In [26]:
response.messages

[{'content': None,
  'role': 'assistant',
  'annotations': None,
  'executed_tools': None,
  'function_call': None,
  'reasoning': 'We need to get task status. Use get_task_status with task_id "1".',
  'tool_calls': [{'id': 'fc_a3d288b3-89a9-43e6-bb63-440379eef3d9',
    'function': {'arguments': '{"task_id":"1"}', 'name': 'get_task_status'},
    'type': 'function'}],
  'sender': 'Task Manager'},
 {'role': 'tool',
  'tool_call_id': 'fc_a3d288b3-89a9-43e6-bb63-440379eef3d9',
  'tool_name': 'get_task_status',
  'content': '{\n  "id": "1",\n  "title": "Fix email validation bug in login page",\n  "description": "The login form is not correctly validating email addresses. This is a critical issue as users cannot log in properly. Implement proper email format validation and update tests accordingly.",\n  "category": "Bug",\n  "priority": "Critical",\n  "assigned_to": "alice",\n  "status": "New",\n  "created_at": "2026-08-11T11:41:24.616485",\n  "updated_at": "2026-08-11T11:41:24.616485",\n  "